# COVID-19 Data Cleaning & Preprocessing Pipeline
**Objective:** Clean raw COVID-19 data, handle missing values, correct data types, and prepare finalized datasets for Power BI visualization.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
# Load raw dataset
print("Loading data...")
df = pd.read_excel('../Data/CovidDeaths.xlsx')

# Initial inspection
df.head()

Loading data...


,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,gdp_per_capita,extreme_poverty,cardiovasc_death_rate,diabetes_prevalence,female_smokers,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index
0,AFG,Asia,Afghanistan,2020-02-24,1.0,1.0,NaN,NaN,NaN,NaN,...,1803.987,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511
1,AFG,Asia,Afghanistan,2020-02-25,1.0,0.0,NaN,NaN,NaN,NaN,...,1803.987,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511
2,AFG,Asia,Afghanistan,2020-02-26,1.0,0.0,NaN,NaN,NaN,NaN,...,1803.987,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511
3,AFG,Asia,Afghanistan,2020-02-27,1.0,0.0,NaN,NaN,NaN,NaN,...,1803.987,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511
4,AFG,Asia,Afghanistan,2020-02-28,1.0,0.0,NaN,NaN,NaN,NaN,...,1803.987,NaN,597.029,9.59,NaN,NaN,37.746,0.5,64.83,0.511


## 1. Advanced Data Cleaning & Imputation
- Remove duplicates and parse dates.
- Enforce numeric types to prevent calculation errors.
- Handle missing values gracefully (impute 0 for unrecorded cases/metrics).
- Filter valid countries and dynamically drop highly null columns to optimize performance.

In [6]:
print("Starting deep data cleaning and preprocessing...")

# 1. Remove duplicate rows if any exist
df.drop_duplicates(inplace=True)

# 2. Standardize datetime format for time-series analysis
df['date'] = pd.to_datetime(df['date'])

# 3. Enforce numeric data types for critical columns (coerce unparseable data to NaN)
cols_to_numeric = ['new_cases', 'total_cases', 'new_deaths', 'total_deaths', 'population']
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# =====================================================================
# ADVANCED MISSING VALUES HANDLING (MASTER LEVEL)
# =====================================================================

# 4.1. Fill missing core metrics with 0 (Assuming NaN means 0 reported cases/deaths)
core_metrics = ['new_cases', 'total_cases', 'new_deaths', 'total_deaths']
df[core_metrics] = df[core_metrics].fillna(0)

# 4.2. Fill missing medical & vaccination metrics with 0 
# (Prevents 'Blank' calculation errors in Power BI DAX)
medical_metrics = [
    'icu_patients', 'hosp_patients', 'total_tests', 'new_tests', 
    'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'new_vaccinations'
]
for col in medical_metrics:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 4.3. Drop rows with missing or zero Population
# (Population is strictly required as the denominator for per-capita rate calculations)
df = df.dropna(subset=['population'])
df = df[df['population'] > 0]

# 5. Filter valid countries (Exclude aggregated macro-regions like 'World' or Income levels)
df_valid_continent = df[df['continent'].notna()].copy()

# 6. Optimize file size: Dynamically drop columns with more than 80% missing data
# (Keep only columns that have at least 20% non-null values)
threshold = len(df_valid_continent) * 0.2 
df_valid_continent = df_valid_continent.dropna(thresh=threshold, axis=1)

print(f"Data shape after advanced cleaning: {df_valid_continent.shape}")

Starting deep data cleaning and preprocessing...
Data shape after advanced cleaning: (80960, 48)


## 2. Export Master Dataset
Export the finalized, highly optimized dataset to a CSV file for high-speed ingestion into Power BI.

In [7]:
# =========================================================
# EXPORT MASTER FILE (CSV FORMAT FOR OPTIMAL PERFORMANCE)
# =========================================================

# Define the export directory using pathlib for cross-platform compatibility
export_path = Path('../Data/clean_data')
export_path.mkdir(parents=True, exist_ok=True)

# Export as CSV for significantly faster Power BI loading times compared to Excel
master_file_path = export_path / 'CovidDeaths_Cleaned.csv'
df_valid_continent.to_csv(master_file_path, index=False)

print(f"Master File successfully exported to: {master_file_path}")

Master File successfully exported to: ..\Data\clean_data\CovidDeaths_Cleaned.csv
